In [0]:
from pathlib import Path
import sys

current = Path.cwd().resolve()
repo_root = next(
    (
        path for path in (current, *current.parents)
        if (path / "transformations" / "silver").is_dir()
        and (path / "sources" / "reference").is_dir()
    ),
    None,
)

if repo_root is None:
    raise RuntimeError("No se encontró la raíz del repositorio.")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import transformations.silver.dummyjson_transform as dummyjson

from transformations.silver.dummyjson_transform import (
    read_dummyjson_products,
    normalize_dummyjson_products,
    validate_dummyjson_products,
    write_dummyjson_delta,
)

ingestion_run_id = "172c6634-a566-4ae8-bfc7-04ae228dae77"

bronze_root = (
    "abfss://bronze@starlpdevdiego01.dfs.core.windows.net/"
    "dummyjson/products/ingestion_date=2026-10-11/"
    f"run_id={ingestion_run_id}"
)

raw_products = read_dummyjson_products(spark, bronze_root)

product_count = raw_products.count()
file_count = raw_products.select("source_file").distinct().count()

print("Productos:", product_count)
print("Archivos con productos:", file_count)

assert product_count == 38, "Se esperaban 38 productos"
assert file_count == 9, "Se esperaban nueve archivos"

In [0]:
from datetime import datetime, timezone
from uuid import uuid4

processing_run_id = str(uuid4())
processed_at = datetime.now(timezone.utc)

normalized_products = dummyjson.normalize_dummyjson_products(
    raw_products,
    ingestion_run_id,
    processing_run_id,
    processed_at,
)

checks = dummyjson.validate_dummyjson_products(normalized_products)

assert normalized_products.count() == 38, "Conteo diferente al normalizar"

print("CONTROLES:", checks)

display(
    normalized_products.select(
        "external_product_id",
        "external_product_name",
        "external_price",
        "external_currency_code",
        "source_system",
    ).orderBy("external_product_id").limit(5)
)

In [0]:
silver_root = (
    "abfss://silver@starlpdevdiego01.dfs.core.windows.net"
)

result = dummyjson.write_dummyjson_delta(
    spark,
    normalized_products,
    silver_root,
)

print("RESULTADO:", result)
print("Processing run ID:", processing_run_id)

In [0]:
spark.sql("""
    CREATE SCHEMA IF NOT EXISTS
    dbw_arlp_dev_diego01.silver_external
""")

spark.sql("""
    CREATE TABLE IF NOT EXISTS
    dbw_arlp_dev_diego01.silver_external.dummyjson_products
    USING DELTA
    LOCATION 'abfss://silver@starlpdevdiego01.dfs.core.windows.net/external/dummyjson_products'
""")

spark.sql("""
    SELECT COUNT(*) AS total_products
    FROM dbw_arlp_dev_diego01.silver_external.dummyjson_products
""").show()